In [1]:
import torch
from torch.profiler import ProfilerActivity, profile
from transformers import AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"


def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def call(model, input_ids):
    return model.generate(input_ids, attention_mask=torch.ones_like(input_ids), max_new_tokens=20, do_sample=False)


def manual_params(config):
    vocab = config.vocab_size
    hidden = config.hidden_size
    layers = config.num_hidden_layers
    heads = config.num_attention_heads
    kv_heads = config.num_key_value_heads
    inter = config.intermediate_size
    head_dim = getattr(config, "head_dim", hidden // heads)

    layer = (heads * head_dim) * hidden + 2 * (kv_heads * head_dim) * hidden + hidden * (heads * head_dim)
    layer += 3 * inter * hidden + 2 * hidden + 2 * head_dim
    total = vocab * hidden + layers * layer + hidden

    if not getattr(config, "tie_word_embeddings", True):
        total += vocab * hidden
    return total


def top(events, key, limit=20):
    for event in sorted(events, key=lambda item: getattr(item, key), reverse=True)[:limit]:
        print(f"{event.key}\t{getattr(event, key) / 1000:.2f} ms\tcount={event.count}")

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()

config = model.config
input_ids = torch.randint(0, config.vocab_size, (1, 1000), device=device)

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Device: cuda
GPU: NVIDIA GeForce RTX 4070 Ti


In [3]:
print(f"Manual param count: {manual_params(config)}")
print(f"PyTorch param count: {sum(param.numel() for param in model.parameters())}")

Manual param count: 4022468096
PyTorch param count: 4022468096


In [4]:
for name, param in model.named_parameters():
    print(f"{name:50} {str(list(param.shape)):20} {param.numel()}")

model.embed_tokens.weight                          [151936, 2560]       388956160
model.layers.0.self_attn.q_proj.weight             [4096, 2560]         10485760
model.layers.0.self_attn.k_proj.weight             [1024, 2560]         2621440
model.layers.0.self_attn.v_proj.weight             [1024, 2560]         2621440
model.layers.0.self_attn.o_proj.weight             [2560, 4096]         10485760
model.layers.0.self_attn.q_norm.weight             [128]                128
model.layers.0.self_attn.k_norm.weight             [128]                128
model.layers.0.mlp.gate_proj.weight                [9728, 2560]         24903680
model.layers.0.mlp.up_proj.weight                  [9728, 2560]         24903680
model.layers.0.mlp.down_proj.weight                [2560, 9728]         24903680
model.layers.0.input_layernorm.weight              [2560]               2560
model.layers.0.post_attention_layernorm.weight     [2560]               2560
model.layers.1.self_attn.q_proj.weight         

In [5]:
# Warm-up
print("Running warm-up...")
for _ in range(3):
    call(model, input_ids)
sync()

# CPU Profiling
print("\n--- CPU Profiling ---")
with torch.no_grad():
    with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as cpu_prof:
        call(model, input_ids)

cpu_prof.export_chrome_trace("trace_cpu.json")
top(cpu_prof.key_averages(), "cpu_time_total")

# GPU Profiling
if torch.cuda.is_available():
    print("\n--- GPU Profiling ---")
    with torch.no_grad():
        with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as gpu_prof:
            call(model, input_ids)

    gpu_prof.export_chrome_trace("trace_gpu.json")
    events = gpu_prof.key_averages()

    print("\nGPU Top Activities:")
    top(events, "device_time_total")
    print("\nGPU Self Top Activities:")
    top(events, "self_device_time_total")

Running warm-up...

--- CPU Profiling ---


/home/student/.local/lib/python3.10/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


cudaLaunchKernel	106.10 ms	count=41917
aten::linear	85.84 ms	count=5060
aten::is_nonzero	83.99 ms	count=42
aten::item	83.95 ms	count=42
aten::_local_scalar_dense	83.91 ms	count=42
cudaStreamSynchronize	83.65 ms	count=45
aten::matmul	71.53 ms	count=5080
aten::to	59.89 ms	count=8977
aten::_to_copy	55.40 ms	count=5975
aten::mm	52.17 ms	count=5060
aten::mul	47.25 ms	count=9480
aten::scaled_dot_product_attention	34.03 ms	count=720
aten::add	33.17 ms	count=5840
aten::copy_	33.10 ms	count=6036
aten::_scaled_dot_product_flash_attention	31.10 ms	count=720
aten::mean	27.32 ms	count=2900
aten::_flash_attention_forward	25.09 ms	count=720
aten::pow	19.31 ms	count=2900
aten::cat	16.42 ms	count=2960
aten::transpose	15.72 ms	count=12892

--- GPU Profiling ---

GPU Top Activities:
aten::matmul	449.75 ms	count=5080
aten::linear	449.73 ms	count=5060
aten::mm	449.73 ms	count=5060
std::enable_if<!(false), void>::type internal::gemvx::kernel<int, int, __half, __half, __half, float, false, true, true, false,